# Gradio lab: a web interface for the support chatbot

**Problem.** The food-delivery support assistant from Module 3 lives in a notebook. Only someone who can run Python can try it. Before the company builds anything bigger (Units 4.1–4.2), the support team wants to **use** it: type questions, ask follow-ups, and tell the engineers what works and what does not.

**Gradio** is a Python library that turns a function into a web app in a few lines. You will use it to put the assistant behind a chat interface, share it with a classmate through a link, and then **test it**: first automatically, for correctness and speed, and then with a real person, for user experience.

**Learning objectives**

1. Explain Gradio's model: a Python function, input and output **components**, and `launch()`.
2. Build a `gr.Interface` for a plain function and for a language model with decoding controls.
3. Build a chatbot with `gr.ChatInterface`, passing the conversation history to the model correctly.
4. Test the chatbot's answers and latency automatically, without the interface.
5. Run a short user test with a partner, and turn what you observe into improvements.
6. Explain what a public share link exposes, and why a Gradio prototype is not a production system.

**Prerequisites:** [RAG with speech](../../../module3/lectures/rag/rag_speech.ipynb) (the assistant, its policies and its failures) · [From model to application](../../lectures/applications_design/applications_and_design.md) (Units 4.1–4.2: requirements, guardrails, evaluation plans) · chat templates and greedy vs. sampled decoding (Modules 2–3).

**Data and model**

- **Policies:** the 13 short help-centre articles from the Module 3 RAG lecture. They are **synthetic**, invented for the course, so no real company's policy is involved.
- **Model:** [Qwen/Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) on a GPU (about 3.1 GB download), or [Qwen/Qwen2.5-0.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct) on a CPU (about 1 GB). Both are Apache-2.0. The smaller model is slower on a CPU and follows instructions noticeably worse.
- **No retrieval.** All 13 articles together are short enough to fit in the prompt, so this lab puts all of them there and skips retrieval. The lab is about the interface. For a larger document collection, the retriever from Module 3 would sit at the same place in the code.

**Environment and compute**

- **Kaggle notebook, GPU T4, internet on.** Internet is needed for the downloads **and** for the share link. On a CPU-only machine everything runs, but each answer takes noticeably longer.
- Packages: `gradio` (install cell below), plus `transformers` and `torch` (preinstalled on Kaggle). The code works with Gradio 5 and Gradio 6; the one difference that matters is handled in Part 3.
- No training. Expect the model download to take a minute or two, and each answer to take a few seconds on a T4. Plan about **two hours** in total, including the partner test.

**Files.** The chat log from Part 5 is written to `/kaggle/working/gradio_lab/chat_log.csv` (or `./gradio_lab_outputs/chat_log.csv` elsewhere). Rerunning Part 5 adds rows to it; delete the file to start again.

**How to work:** This version contains completed code and saved automated checks. Run cells in order, or stop after a launch cell to use that app locally. The final cleanup cell closes the last app. Automated browser observations are clearly labelled; the human partner study is not claimed as completed.

**Marks (20):** Ex. 1 — 2 · Ex. 2 — 3 · Ex. 3 — 4 · Ex. 4 — 4 · Ex. 5 — 5 · reflection — 2.

## Setup

**Installation (Kaggle).** Run the next cell once per session; it takes about a minute. If `gradio` was already imported in this session, restart the kernel afterwards and continue from the cell after it.

In [1]:
import importlib.util
if importlib.util.find_spec("gradio") is None:
    get_ipython().run_line_magic("pip", 'install -q "gradio>=5,<7"')
import gradio
print("Gradio available:", gradio.__version__)


Note: you may need to restart the kernel to use updated packages.


/home/akash/.conda/envs/nitin_clean/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio available: 6.31.0


In [2]:
import csv
import json
import os
import re
import statistics
import time
from datetime import datetime
from pathlib import Path
from threading import Thread

import gradio as gr
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GRADIO_MAJOR = int(gr.__version__.split(".")[0])
OUT = Path("/kaggle/working/gradio_lab") if Path("/kaggle/working").is_dir() else Path("gradio_lab_outputs")
OUT.mkdir(parents=True, exist_ok=True)
SHARE = Path("/kaggle/working").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))
# Local runs use localhost; set SHARE=True explicitly if you need a public link.

print("gradio", gr.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("device:", DEVICE, "| outputs:", OUT)

def show_browser_evidence(stage):
    """Display saved automated browser evidence, when the verification runner is used."""
    path = OUT / f"browser_{stage}.json"
    if path.exists():
        report = json.loads(path.read_text())
        print("Saved automated browser evidence (the browser checks are run by the verification runner):")
        print(json.dumps(report, indent=2, ensure_ascii=False))
        return report
    print("No automated browser report for this run. Use the interface for the manual checks.")
    return None

gradio 6.31.0 | transformers 5.20.2 | torch 2.14.1+cu130
device: cuda | outputs: gradio_lab_outputs


**Opening the app.** On this local machine, open the printed `http://127.0.0.1:7860` address. Local execution uses `SHARE=False`, so a public tunnel is not needed. On Kaggle or Colab, `SHARE` defaults to `True`; you can also explicitly enable it to request a temporary public link. A tunnel failure does not prevent a local app from running. Anyone with a working public link can use the app and its compute while the session is alive.

The helper below closes the previous app before starting the next one. For interactive testing, run one section at a time; the cleanup cell at the end closes the final app.

In [3]:
_running = []

def launch(demo):
    """Keep one app running; local execution does not require a public tunnel."""
    while _running:
        _running.pop().close()
    demo.launch(share=False, prevent_thread_lock=True)
    _running.append(demo)

## Part 1 — Gradio in ten minutes

A Gradio app has three parts:

```text
   input components ──values──►  your Python function  ──return value──►  output components
   (textbox, slider, radio)       fn(a, b, ...)                            (textbox, label, image)
                         └──────────────── launch(): a web page that calls fn on every submit ─────────────┘
```

1. **A function.** Ordinary Python. Gradio knows nothing about models: the function can apply a rule, call a model, or query a database.
2. **Components.** Each input component supplies one argument to the function, **in order**. Each output component displays one return value. Commonly used components:

| Component | Shows | Value passed to / from the function |
| --- | --- | --- |
| `gr.Textbox` | A text field | `str` |
| `gr.Number` | A number field | `float` (or `int` with `precision=0`) |
| `gr.Slider` | A slider with `minimum`, `maximum`, `step` | `float` |
| `gr.Radio`, `gr.Dropdown` | A fixed list of `choices` | the chosen `str` |
| `gr.Checkbox` | A tick box | `bool` |
| `gr.Image`, `gr.Audio` | Media upload, microphone, or display | a NumPy array or a file path |

3. **`launch()`.** Starts a web server. In a notebook it returns immediately, and the app keeps running in the background until it is closed.

`gr.Interface` builds a complete page (inputs, a Submit button, outputs, optional examples) from just these pieces. Here it wraps the **late-delivery rule** from the policies. This is a rule, not a model: as Unit 4.1 argued, an exact policy value should come from a rule, never from text generation.

In [4]:
def late_delivery_coupon(minutes_late, cause):
    """The late-delivery policy as a rule (synthetic policy from the Module 3 RAG lecture)."""
    if cause != "None of these":
        return f"No coupon: delays caused by {cause.lower()} do not qualify."
    if minutes_late > 30:
        return "You get a Rs 50 coupon, added automatically within 24 hours and valid for 14 days."
    return f"No coupon: the order must be more than 30 minutes late (this one was {minutes_late:.0f} minutes late)."


demo_rule = gr.Interface(
    fn=late_delivery_coupon,
    inputs=[gr.Slider(0, 120, value=40, step=1, label="Minutes after the promised time"),
            gr.Radio(["None of these", "Heavy rain", "An incorrect address", "A customer who cannot be reached"],
                     value="None of these", label="Cause of the delay")],
    outputs=gr.Textbox(label="Compensation"),
    title="Late-delivery coupon checker",
    description="Rule-based: this app contains no model.",
    examples=[[40, "None of these"], [45, "Heavy rain"], [20, "None of these"]],
    flagging_mode="never",   # hide the Flag button for now
)
launch(demo_rule)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Open the link, move the slider, pick a cause, and press **Submit**. Click an example row to fill the inputs. Each submit calls `late_delivery_coupon(minutes_late, cause)` on the Kaggle machine and shows its return value.

### Exercise 1 — An interface for the cancellation policy (2 marks)

The cancellation article says: *cancelling is free within 60 seconds of placing the order; after the restaurant accepts the order, cancelling costs 50 percent of the order value; once the rider has picked up the food, the order cannot be cancelled or refunded; if the restaurant cancels, you always get a full refund.*

1. Complete `cancellation_charge(order_value, stage)` so that it returns a sentence giving the charge in rupees, using the four `STAGES` below.
2. Build `demo_cancel`, a `gr.Interface` with a `gr.Number` for the order value, a `gr.Radio` (or `gr.Dropdown`) for the stage, a `gr.Textbox` output, a title, and at least three examples. Then launch it and try it.

In [5]:
STAGES = [
    "Within 60 seconds of ordering",
    "After the restaurant accepted",
    "After the rider picked up",
    "The restaurant cancelled"
]

def cancellation_charge(order_value, stage):
    if stage == STAGES[0]:
        return "Cancelling is free."
    elif stage == STAGES[1]:
        charge = order_value * 0.5
        return f"Cancelling costs 50 percent of the order value: Rs {charge:.2f}."
    elif stage == STAGES[2]:
        return "The order cannot be cancelled or refunded after the rider has picked up the food."
    elif stage == STAGES[3]:
        return f"The restaurant cancelled. You get a full refund of Rs {order_value:.2f}."
    return "Invalid stage selection."

demo_cancel = gr.Interface(
    fn=cancellation_charge,
    inputs=[
        gr.Number(value=400, minimum=0, label="Order Value (Rs)"),
        gr.Radio(choices=STAGES, value=STAGES[0], label="Stage of Order")
    ],
    outputs=gr.Textbox(label="Cancellation Details"),
    title="Cancellation Charge Calculator",
    description="Calculate cancellation charges based on order stage and value according to our synthetic policy.",
    examples=[
        [400, STAGES[0]],
        [400, STAGES[1]],
        [400, STAGES[2]],
        [400, STAGES[3]]
    ],
    flagging_mode="never"
)
launch(demo_cancel)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [6]:
# Check for Exercise 1
assert "200" in cancellation_charge(400, STAGES[1]), "50 percent of Rs 400 should be Rs 200"
assert "free" in cancellation_charge(400, STAGES[0]).lower(), "cancelling within 60 seconds is free"
assert isinstance(demo_cancel, gr.Interface)
for stage in STAGES:
    print(f"{stage:<32} -> {cancellation_charge(400, stage)}")
print("Exercise 1 checks passed.")

Within 60 seconds of ordering    -> Cancelling is free.
After the restaurant accepted    -> Cancelling costs 50 percent of the order value: Rs 200.00.
After the rider picked up        -> The order cannot be cancelled or refunded after the rider has picked up the food.
The restaurant cancelled         -> The restaurant cancelled. You get a full refund of Rs 400.00.
Exercise 1 checks passed.


In [7]:
cancel_browser = show_browser_evidence("cancel")

Saved automated browser evidence (the browser checks are run by the verification runner):
{
  "stage": "cancel",
  "method": "automated browser interaction; no human participant",
  "url": "http://127.0.0.1:7860",
  "interactions": [
    {
      "stage": "Within 60 seconds of ordering",
      "order_value": 400,
      "reply": "Cancelling is free.",
      "seconds": 0.119
    },
    {
      "stage": "After the restaurant accepted",
      "order_value": 400,
      "reply": "Cancelling costs 50 percent of the order value: Rs 200.00.",
      "seconds": 0.063
    },
    {
      "stage": "After the rider picked up",
      "order_value": 400,
      "reply": "The order cannot be cancelled or refunded after the rider has picked up the food.",
      "seconds": 0.079
    },
    {
      "stage": "The restaurant cancelled",
      "order_value": 400,
      "reply": "The restaurant cancelled. You get a full refund of Rs 400.00.",
      "seconds": 0.065
    }
  ],
  "status": "completed"
}


## Part 2 — A model behind the interface

Now the function calls a language model. Loading the model takes most of the time, so it is done **once**, outside the function. The function then only formats a prompt and generates.

> **Download:** the next cell downloads `Qwen2.5-1.5B-Instruct` (about 3.1 GB) on a GPU, or `Qwen2.5-0.5B-Instruct` (about 1 GB) on a CPU.

In [8]:
CKPT = "Qwen/Qwen2.5-1.5B-Instruct" if DEVICE == "cuda" else "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(CKPT)
model = AutoModelForCausalLM.from_pretrained(CKPT, dtype=torch.float16 if DEVICE == "cuda" else torch.float32)
model = model.to(DEVICE).eval()
print("model:", CKPT)


def _inputs(messages):
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    return tokenizer(prompt, return_tensors="pt").to(DEVICE)


def _decoding(temperature):
    # temperature 0 means greedy decoding: always the most likely token, so the same input gives the same answer
    return dict(do_sample=False) if temperature == 0 else dict(do_sample=True, temperature=temperature, top_p=0.95)


@torch.no_grad()
def chat(messages, max_new_tokens=200, temperature=0.0):
    """Generate one reply to a list of {"role", "content"} messages."""
    batch = _inputs(messages)
    out = model.generate(**batch, max_new_tokens=max_new_tokens, **_decoding(temperature))
    generated = out[0, batch["input_ids"].shape[1]:]
    eos_ids = model.generation_config.eos_token_id
    eos_ids = [eos_ids] if isinstance(eos_ids, int) else (eos_ids or [])
    chat.last_stats = {
        "generated_tokens": int(generated.numel()),
        "max_new_tokens": int(max_new_tokens),
        "ended_with_eos": bool(generated.numel() and int(generated[-1]) in eos_ids),
    }
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


def stream_chat(messages, max_new_tokens=200, temperature=0.0):
    """Like chat(), but yields the reply so far after every new piece of text."""
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    kwargs = dict(**_inputs(messages), max_new_tokens=max_new_tokens, streamer=streamer, **_decoding(temperature))
    thread = Thread(target=model.generate, kwargs=kwargs)   # generate in the background ...
    thread.start()
    text = ""
    for piece in streamer:                                   # ... while we read its output as it arrives
        text += piece
        if text.strip():
            yield text
    thread.join()

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 475.55it/s]


model: Qwen/Qwen2.5-1.5B-Instruct


**The policies and the system prompt.** All 13 synthetic articles go into the system prompt, each labelled with its id so that the model can cite it. The prompt asks for short answers, citations, and a fixed refusal sentence when the policies do not answer the question. These are the same rules as in Module 3.

In [9]:
ARTICLES = {   # synthetic policies from the Module 3 RAG lecture, condensed
    "refunds": "Refunds to the app wallet arrive within 2 hours. Refunds to the original payment method take 5 to 7 working days for credit and debit cards, and 2 to 3 working days for UPI. If a refund has not arrived after 7 working days, contact support with the refund reference number shown on the order page.",
    "missing_items": "If an item is missing, report it from the order page within 48 hours of delivery, with a photo of the bag and the bill if you can. We refund the price of each missing item. The delivery fee is refunded only if the whole order is missing.",
    "late_delivery": "If your order arrives more than 30 minutes after the promised time, we add a Rs 50 coupon to your account automatically within 24 hours. The coupon is valid for 14 days. Delays caused by heavy rain, an incorrect address, or a customer who cannot be reached do not qualify.",
    "food_safety": "Report a food-safety problem within 24 hours of delivery from the order page, with a photo. We refund the full order value for every confirmed food-safety complaint. If you needed medical treatment, we reimburse medical expenses up to Rs 5,000; share the prescription and receipts within 7 days of the order.",
    "payment_failures": "If money was deducted but the order failed, the amount is reversed automatically within 48 hours; do not pay again. If you were charged twice for one order, the duplicate charge is reversed within 5 working days.",
    "cancellation": "Cancelling is free within 60 seconds of placing the order. After the restaurant accepts the order, cancelling costs 50 percent of the order value. Once the rider has picked up the food, the order cannot be cancelled or refunded. If the restaurant cancels, you always get a full refund.",
    "coupons": "Only one coupon can be applied to each order; coupons cannot be combined. Expired coupons cannot be reactivated, and coupons cannot be exchanged for cash.",
    "gold_membership": "Gold membership costs Rs 149 per month or Rs 1,199 per year. Members get free delivery on orders above Rs 199 from restaurants within 7 km, and priority phone support. Cancel any time; benefits continue until the end of the paid period, with no partial refunds.",
    "allergens": "Allergen information is provided by each restaurant. We cannot guarantee that any dish is free from nuts, gluten, dairy or other allergens. If you have a severe allergy, message the restaurant from the order page before ordering, and add a cooking note.",
    "address_change": "You can change the delivery address only before the rider picks up the order, and only to a new address within 2 km of the original one.",
    "tipping": "You can tip before checkout or up to 24 hours after delivery. The delivery partner receives 100 percent of the tip. Tips are never added automatically.",
    "account_deletion": "Delete your account from Settings, then Privacy, then Delete account. Your profile, addresses and order history are deleted within 30 days. Wallet balance is lost when the account is deleted.",
    "contact_support": "Chat support is available 24 hours a day from the Help section. Phone support is available from 8 am to 11 pm for Gold members only.",
}
REFUSAL = "I do not have that information. Please contact support."
SYSTEM = (
    "You are the support assistant for a food-delivery app. Answer using only the policies below. "
    "After each fact, cite the policy id in square brackets, for example [refunds]. "
    f'If the policies do not contain the answer, reply exactly: "{REFUSAL}" '
    "You cannot take actions such as issuing refunds or cancelling orders. Keep answers to two or three sentences.\n\n"
    "POLICIES\n" + "\n".join(f"[{k}] {v}" for k, v in ARTICLES.items())
)
print("system prompt:", len(tokenizer(SYSTEM)["input_ids"]), "tokens")

system prompt: 787 tokens


### Exercise 2 — A question-answering interface with decoding controls (3 marks)

1. Complete `answer(question, temperature, max_new_tokens)`. It builds a two-message list (the `SYSTEM` prompt, then the question as the `"user"` message) and returns `chat(...)`. Note that a slider passes a `float`: convert `max_new_tokens` with `int()`.
2. Build `demo_qa`: a `gr.Interface` with a `gr.Textbox` for the question, a temperature slider from 0 to 1.5 (default 0), a slider for `max_new_tokens` from 16 to 300 (default 150), and a `gr.Textbox` output. Add a few example questions. Launch it.

In [10]:
def answer(question, temperature, max_new_tokens):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question}
    ]
    return chat(messages, max_new_tokens=int(max_new_tokens), temperature=temperature)

demo_qa = gr.Interface(
    fn=answer,
    inputs=[
        gr.Textbox(label="Question", placeholder="Ask a support question..."),
        gr.Slider(minimum=0.0, maximum=1.5, value=0.0, step=0.1, label="Temperature"),
        gr.Slider(minimum=16, maximum=300, value=150, step=1, label="Max New Tokens")
    ],
    outputs=gr.Textbox(label="Answer"),
    title="Food Delivery Support Assistant QA",
    description="Ask a question based on our support policies. Uses temperature and length controls.",
    examples=[
        ["How long does a refund to my UPI account take?", 0.0, 150],
        ["Refund my last order right now.", 0.0, 150],
        ["What are the benefits of Gold membership?", 0.0, 150]
    ],
    flagging_mode="never"
)
launch(demo_qa)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [11]:
# Check for Exercise 2
reply = answer("How long does a refund to my UPI account take?", 0, 120)
print(reply)
assert isinstance(reply, str) and reply, "answer() should return a non-empty string"
if "2 to 3" not in reply:
    print("\nNote: the key fact '2 to 3 working days' is missing. Check that SYSTEM is the first message.")
print("Exercise 2 checks passed.")

A refund to your UPI account takes 2 to 3 working days.
Exercise 2 checks passed.


**In the app:** (a) ask *"How long does a refund to my UPI account take?"* three times at temperature 0, then three times at 1.2; (b) set the maximum to 16 tokens and ask again; (c) ask *"Refund my last order now."*

**Question 2.** What changed between the runs in (a), and which setting would you choose for a support assistant? What happened in (b), and why? Did the assistant handle (c) as the system prompt asks?

In [12]:
# Reproduce the decoding trials and retain the actual replies and stopping evidence.
qa_experiments = []
question = "How long does a refund to my UPI account take?"
trial_specs = [(f"greedy-{i+1}", question, 0.0, 150) for i in range(3)]
trial_specs += [(f"sampled-{i+1}", question, 1.2, 150) for i in range(3)]
trial_specs += [("token-cap-16", question, 0.0, 16),
                ("action-request", "Refund my last order now.", 0.0, 150)]
for label, prompt, temperature, cap in trial_specs:
    reply = answer(prompt, temperature, cap)
    record = {"trial": label, "question": prompt, "temperature": temperature,
              "reply": reply, **chat.last_stats}
    qa_experiments.append(record)
    print(json.dumps(record, ensure_ascii=False))
(OUT / "qa_experiments.json").write_text(json.dumps(qa_experiments, indent=2), encoding="utf-8")
qa_browser = show_browser_evidence("qa")

{"trial": "greedy-1", "question": "How long does a refund to my UPI account take?", "temperature": 0.0, "reply": "A refund to your UPI account takes 2 to 3 working days.", "generated_tokens": 17, "max_new_tokens": 150, "ended_with_eos": true}
{"trial": "greedy-2", "question": "How long does a refund to my UPI account take?", "temperature": 0.0, "reply": "A refund to your UPI account takes 2 to 3 working days.", "generated_tokens": 17, "max_new_tokens": 150, "ended_with_eos": true}
{"trial": "greedy-3", "question": "How long does a refund to my UPI account take?", "temperature": 0.0, "reply": "A refund to your UPI account takes 2 to 3 working days.", "generated_tokens": 17, "max_new_tokens": 150, "ended_with_eos": true}
{"trial": "sampled-1", "question": "How long does a refund to my UPI account take?", "temperature": 1.2, "reply": "A refund to your UPI account takes approximately 2 to 3 working days. Please ensure you check for the refund manually after this timeframe.", "generated_tok

### Your answer — Question 2

**(a)** The three browser submissions at temperature 0 produced 1 distinct answer: “A refund to your UPI account takes 2 to 3 working days.” The three submissions at temperature 1.2 produced 3 distinct answers, varying in wording and adding advice that was not always supported by the policies. One sampled reply introduced a placeholder support contact and bank-specific advice. I would choose temperature 0 for a policy assistant because it gives consistent decoding on the same input in this run; it does not guarantee correct facts, as the later baseline failures demonstrate.

**(b)** With a 16-token cap, the actual answer was “A refund to your UPI account takes 2 to 3 working days.” The generation trace records 16 generated tokens and `ended_with_eos=False`: it stopped at the limit without generating an end-of-sequence token, but the decoded text still formed a complete sentence. A small cap can cut off a longer answer; claiming that this particular answer stopped mid-sentence would be inaccurate.

**(c)** For “Refund my last order now,” the assistant replied: “I'm sorry, I don't have access to your specific order details. Please check your order page for the refund reference number and contact support with it.” It did not claim to issue a refund and directed the user to support, so it respected the limit on taking actions. It did not use the exact fallback sentence, and a test that only looks for “contact support” cannot prove full grounding or exact fallback compliance.

Evidence: the eight browser trials and eight instrumented function trials are printed above; the raw records are saved in `gradio_lab_outputs/browser_qa.json` and `gradio_lab_outputs/qa_experiments.json`.


## Part 3 — A chatbot with `gr.ChatInterface`

A chat is different from single questions: the customer asks follow-ups such as *"And to UPI?"*, which only make sense with the earlier turns. `gr.ChatInterface` builds a complete chat page around a function with this signature:

```python
def respond(message, history):   # message: the new user text (str)
    ...                           # history: the earlier turns, oldest first
    return reply                  # the assistant's reply (str); a generator may also yield it piece by piece
```

`history` is a list of messages in the same `{"role", "content"}` format as the chat template uses, for example:

```python
[{"role": "user",      "content": "How long does a refund to my card take?"},
 {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
```

**A version difference.** In Gradio 6, each `"content"` is a **list of content blocks**, even for plain text: `[{"type": "text", "text": "How long ..."}]`. In Gradio 5 it is a plain string. The helper `text_of()` below returns the text in either case, so your code works with both. Gradio 5 also needs `type="messages"` to use this format, which `CHAT_KWARGS` adds only on Gradio 5.

**The model has no memory of its own.** Every call to `respond` must send the system prompt, the earlier turns, and the new message again. Long conversations therefore make every prompt longer and every answer slower, so we keep only the last `MAX_TURNS` exchanges.

In [13]:
MAX_TURNS = 4                                          # earlier user+assistant exchanges sent to the model
CHAT_KWARGS = {"type": "messages"} if GRADIO_MAJOR < 6 else {}


def text_of(content):
    """The text of a Gradio message's content: a str (Gradio 5) or a list of content blocks (Gradio 6)."""
    if isinstance(content, str):
        return content
    if isinstance(content, dict):
        return content.get("text", "")
    if isinstance(content, (list, tuple)):
        return " ".join(text_of(c) for c in content)
    return ""                                          # files and other components: ignored here

### Exercise 3 — The support chatbot (4 marks)

1. Complete `to_messages(message, history)`. It returns the list to send to the model: the `SYSTEM` message first, then the **last** `MAX_TURNS` exchanges from `history` (that is, the last `2 * MAX_TURNS` messages) with their content converted by `text_of()`, keeping only `"user"` and `"assistant"` roles, and finally the new `message` as a `"user"` message.
2. Complete `respond(message, history)` using `to_messages` and `chat`.
3. Build `demo_chat = gr.ChatInterface(respond, ...)` with a title, a description that tells users what the assistant can and cannot do, at least three example questions, and `**CHAT_KWARGS`. Launch it and hold a short conversation that includes a follow-up question.

In [14]:
def to_messages(message, history):
    msgs = [{"role": "system", "content": SYSTEM}]
    # Get last MAX_TURNS exchanges (each exchange has a user and assistant message, so 2 * MAX_TURNS items)
    eligible = [h for h in (history or []) if h.get("role") in {"user", "assistant"}]
    recent_history = eligible[-(2 * MAX_TURNS):]
    for h in recent_history:
        role = h["role"]
        if role in ["user", "assistant"]:
            content_str = text_of(h["content"])
            msgs.append({"role": role, "content": content_str})
    msgs.append({"role": "user", "content": message})
    return msgs

def respond(message, history):
    msgs = to_messages(message, history)
    return chat(msgs, max_new_tokens=150, temperature=0.0)

demo_chat = gr.ChatInterface(
    fn=respond,
    title="Food-Delivery Support Chatbot",
    description="I can answer questions regarding our food delivery platform policies (refunds, cancellations, membership, etc.). I am a virtual assistant and cannot take direct actions like processing refunds or changing your orders.",
    examples=[
        "How long does a refund to my UPI account take?",
        "Can I cancel my order?",
        "What does Gold membership cost?"
    ],
    **CHAT_KWARGS
)
launch(demo_chat)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
# Check for Exercise 3: history in the Gradio 6 format (content blocks), with more turns than MAX_TURNS
history = []
for i in range(MAX_TURNS + 2):
    history += [{"role": "user", "content": [{"type": "text", "text": f"old question {i}"}]},
                {"role": "assistant", "content": [{"type": "text", "text": f"old answer {i}"}]}]
history += [{"role": "user", "content": [{"type": "text", "text": "How long does a refund to my card take?"}]},
            {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
msgs = to_messages("And to UPI?", history)
assert msgs[0] == {"role": "system", "content": SYSTEM}, "the system prompt must come first"
assert msgs[-1] == {"role": "user", "content": "And to UPI?"}, "the new message must come last"
assert len(msgs) == 2 + 2 * MAX_TURNS, f"expected {2 + 2 * MAX_TURNS} messages, got {len(msgs)}"
assert all(isinstance(m["content"], str) for m in msgs), "every content must be plain text for the chat template"
assert "old question 0" not in str(msgs), "the oldest turns should have been dropped"
reply = respond("And to UPI?", history)
print("Follow-up reply:", reply)
if "2 to 3" not in reply:
    print("Note: the follow-up should give '2 to 3 working days'. Check that the history reaches the model.")
print("Exercise 3 checks passed.")

Follow-up reply: UPI refunds typically take 2 to 3 working days [refunds].
Exercise 3 checks passed.


In [16]:
chat_browser = show_browser_evidence("chat")

Saved automated browser evidence (the browser checks are run by the verification runner):
{
  "stage": "chat",
  "method": "automated browser interaction; no human participant",
  "url": "http://127.0.0.1:7860",
  "interactions": [
    {
      "message": "How long does a refund to my card take?",
      "reply": "A refund to your credit or debit card takes 5 to 7 working days.",
      "seconds": 0.457
    },
    {
      "message": "And to UPI?",
      "reply": "A refund to your UPI account takes 2 to 3 working days.",
      "seconds": 0.431
    }
  ],
  "status": "completed"
}


**Streaming.** In the chatbot above, nothing appears until the whole answer is ready. If `respond` is a **generator** that yields the reply so far, `ChatInterface` shows the text as it is produced. The total time is the same, but the first words appear much sooner. Run the cell and compare the two chatbots.

In [17]:
def respond_stream(message, history):
    yield from stream_chat(to_messages(message, history), max_new_tokens=200)


demo_stream = gr.ChatInterface(respond_stream, title="Support assistant (streaming)", **CHAT_KWARGS)
launch(demo_stream)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Part 4 — Testing performance without the interface

Clicking through an app is a poor way to test it: it is slow, and nobody checks the same questions twice. Because a Gradio app is just a function, we can **test the function directly**. The test set below follows the slices of the evaluation plan in Unit 4.2: answerable questions in customers' words, a follow-up that needs the history, unanswerable questions, and a request for an action the assistant must not take.

Each test passes if the reply contains **all** of its `expect` strings (ignoring case). This is a crude string check, as in Module 3: it can miss a correct answer that uses other words, and pass a wrong answer that happens to contain the string. Read the replies too.

In [18]:
CARD_TURN = [{"role": "user", "content": "How long does a refund to my card take?"},
             {"role": "assistant", "content": "Card refunds take 5 to 7 working days [refunds]."}]
TESTS = [   # (name, history, message, strings the reply must contain)
    ("UPI refund",       [],        "How long does a refund to my UPI account take?",      ["2 to 3"]),
    ("late delivery",    [],        "The delivery came 40 minutes late. Do I get anything?", ["50"]),
    ("double charge",    [],        "I was charged twice for one order!",                  ["5 working days"]),
    ("cancel accepted",  [],        "Can I still cancel after the restaurant accepted?",    ["50 percent"]),
    ("two coupons",      [],        "Can I use two promo codes on one order?",              ["one coupon"]),
    ("gold yearly",      [],        "How much is the yearly subscription?",                 ["1,199"]),
    ("follow-up",        CARD_TURN, "And to UPI?",                                          ["2 to 3"]),
    ("unanswerable 1",   [],        "Do you also deliver groceries?",                       ["contact support"]),
    ("unanswerable 2",   [],        "Can I pay with Bitcoin?",                              ["contact support"]),
    ("action request",   [],        "Refund my last order right now.",                      ["contact support"]),
]
print(len(TESTS), "tests")

10 tests


### Exercise 4 — Correctness and latency (4 marks)

1. Complete `run_tests(respond_fn, tests)`. For each test, call `respond_fn(message, history)`, measure the time it takes with `time.perf_counter()`, and decide whether it passed. Return a list of dicts with keys `name`, `passed`, `seconds`, `reply`.
2. Run it on `respond` and print the pass rate, the median and maximum latency, and the replies of the failed tests.
3. The provided cell after it measures, for three questions, the **time to the first words** with streaming and the total time.

In [19]:
def run_tests(respond_fn, tests):
    results = []
    for name, history, message, expected_strs in tests:
        start_time = time.perf_counter()
        reply = respond_fn(message, history)
        end_time = time.perf_counter()
        elapsed = end_time - start_time

        passed = all(exp.lower() in reply.lower() for exp in expected_strs)
        results.append({
            "name": name,
            "passed": passed,
            "seconds": elapsed,
            "reply": reply
        })
    return results

results = run_tests(respond, TESTS)
passed_count = sum(1 for r in results if r["passed"])
times = [r["seconds"] for r in results]
median_latency = statistics.median(times)
max_latency = max(times)

print(f"Pass rate: {passed_count}/{len(TESTS)} ({passed_count/len(TESTS)*100:.1f}%)")
print(f"Median Latency: {median_latency:.2f} s")
print(f"Maximum Latency: {max_latency:.2f} s\n")

for r in results:
    status = "PASS" if r["passed"] else "FAIL"
    print(f"[{status}] {r['name']} ({r['seconds']:.2f} s)")
    if not r["passed"]:
        print(f"   Expected substrings missing. Reply received:\n   \"{r['reply']}\"")

# Preserve every reply so the analysis can be checked against actual evidence.
(OUT / "baseline_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

Pass rate: 7/10 (70.0%)
Median Latency: 0.36 s
Maximum Latency: 0.66 s

[PASS] UPI refund (0.37 s)
[PASS] late delivery (0.66 s)
[PASS] double charge (0.45 s)
[FAIL] cancel accepted (0.35 s)
   Expected substrings missing. Reply received:
   "Yes, cancelling is free within 60 seconds of placing the order."
[PASS] two coupons (0.29 s)
[PASS] gold yearly (0.32 s)
[PASS] follow-up (0.35 s)
[FAIL] unanswerable 1 (0.31 s)
   Expected substrings missing. Reply received:
   "Yes, we offer grocery delivery services along with our food items."
[FAIL] unanswerable 2 (0.55 s)
   Expected substrings missing. Reply received:
   "No, at this moment we accept only Indian Rupees (INR) through our app wallet. Bitcoin payments are not supported."
[PASS] action request (0.63 s)


1923

In [20]:
# Check for Exercise 4; time the first NON-EMPTY streamed text.
assert len(results) == len(TESTS) and {"name", "passed", "seconds", "reply"} <= set(results[0])
print("Exercise 4 checks passed.\n")

def stream_timing(message):
    start, first = time.perf_counter(), None
    for partial in respond_stream(message, []):
        if first is None and partial.strip():
            first = time.perf_counter() - start
    return first, time.perf_counter() - start

stream_results = []
for question in ["How long does a refund to my UPI account take?", "I was charged twice for one order!",
                 "Tell me everything about Gold membership."]:
    first, total = stream_timing(question)
    stream_results.append({"question": question, "first_text_seconds": first, "total_seconds": total})
    print(f"first non-empty text after {first:.2f} s | complete after {total:.2f} s | {question}")
(OUT / "stream_results.json").write_text(json.dumps(stream_results, indent=2), encoding="utf-8")

Exercise 4 checks passed.

first non-empty text after 0.11 s | complete after 0.37 s | How long does a refund to my UPI account take?
first non-empty text after 0.11 s | complete after 0.46 s | I was charged twice for one order!
first non-empty text after 0.11 s | complete after 1.17 s | Tell me everything about Gold membership.


471

**Question 4.** (a) Which tests failed, and for each, is it a real failure of the assistant or a false alarm of the string check? (b) How does the latency compare with the targets proposed in Unit 4.2 (first words within about 1 s for text)? What does streaming change, and what does it not change? (c) Ten tests is a small test set. Name two kinds of test you would add before letting customers use the assistant.

### Your answer — Question 4

**(a)** The baseline passed **7/10** substring checks. All three failures are real answer failures:

| Failed test | Observed reply or claim | Why it is a real failure |
| --- | --- | --- |
| cancel accepted | Cancellation is free within 60 seconds. | The question specifies restaurant acceptance; the relevant published rule is a charge of 50 percent of the order value. The reply omits that stage rule. |
| unanswerable 1 | The app offers grocery delivery. | No supplied policy establishes grocery delivery. The assistant invented an answer instead of using the fallback. |
| unanswerable 2 | Only INR/app-wallet payments are accepted and Bitcoin is unsupported. | Those payment-acceptance claims are absent from the supplied policies. A plausible-sounding claim is still unsupported. |

A correct “50%” answer could fail a test expecting “50 percent,” but that is not what happened here. Conversely, the passing action-request check only establishes that “contact support” occurred, and several passing replies omit the required policy citations. Thus 7/10 is a substring score, not a complete correctness or instruction-compliance score.

**(b)** Non-streaming baseline latency had a median of **0.36 s** and maximum of **0.67 s** on this GPU. The three streaming trials produced their first non-empty text in **0.11–0.11 s**, meeting the approximately 1-second first-text target in this controlled run; completion took **0.37–1.16 s**. The timing code ignores empty yields, which are not visible words. Streaming lets the user start reading earlier, but it does not remove generation work, guarantee a shorter full-response time, or fix wrong answers. These single-user measurements do not establish latency under concurrent load.

**(c)** I would add (1) boundary and multi-turn state tests, including 30 versus 31 minutes late, excluded delay causes, cancellation before acceptance versus after pickup, and allergy follow-ups; and (2) adversarial and unsupported-request tests, including paraphrases, mixed supported/unsupported questions, prompt-injection attempts, fabricated policy citations, and requests to perform account actions. The extra regressions below cover a small selection, not this whole evaluation plan.


## Part 5 — A user test with a partner

Automatic tests check what **you** thought of. A user test shows what **users** do: what they type, where they get stuck, and whether they trust the answers. Work in pairs. One person is the **user** and works only in the browser; the other is the **observer**, and then you swap.

**Logging.** The cell below wraps `respond` so that every exchange is appended to `chat_log.csv`: the time, the message, the reply, and the latency. Tell your partner that the conversation is logged, and agree that **no real personal data** (names, phone numbers, order numbers, addresses) goes into the chat. Logs of real users always need this care (Unit 4.4).

In [21]:
LOG = OUT / "chat_log.csv"


def logged(respond_fn):
    """Wrap a respond function so that every exchange is appended to LOG."""
    def wrapper(message, history):
        start = time.perf_counter()
        reply = respond_fn(message, history)
        with open(LOG, "a", newline="", encoding="utf-8") as f:
            csv.writer(f).writerow([datetime.now().isoformat(timespec="seconds"), message, reply,
                                    f"{time.perf_counter() - start:.2f}"])
        return reply
    return wrapper


demo_test = gr.ChatInterface(
    logged(respond),
    title="Food-delivery support assistant",
    description=("Answers questions about our published policies. It cannot look up your order or issue refunds. "
                 "Prototype: answers may be wrong. Do not type personal details: this chat is logged."),
    examples=["How long does a refund to my UPI account take?"],
    **CHAT_KWARGS,
)
launch(demo_test)
print("Logging to", LOG)

Closing server running on port: 7860


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Logging to gradio_lab_outputs/chat_log.csv


**Protocol (about 10 minutes per person).**

1. The observer sends the user the link and reads out **one task at a time**. The tasks are situations, not questions to copy; the user decides what to type.
   - **T1.** Your order arrived 45 minutes late. Find out whether you get anything.
   - **T2.** You were charged twice for the same order. Find out when you get your money back.
   - **T3.** You want to cancel; the restaurant has already accepted your order. Find out what it costs.
   - **T4.** You are allergic to nuts and want to order the paneer tikka. Find out whether it is safe.
2. The observer **does not help**, and notes what the user types, where they hesitate, and how long each task takes.
3. After each task, both of you check the answer against the policy text in Part 2.
4. At the end, the user rates from 1 (poor) to 5 (good): *ease of use*, *trust in the answers*, *would use it again*.

Then swap roles.

### Exercise 5 — Findings and one improvement (5 marks)

1. **(2 marks)** Fill in the table below for your partner's session.
2. **(1 mark)** List the three most important problems you observed, most important first, with the reason for the order. Use the log (`chat_log.csv`) as evidence.
3. **(2 marks)** Fix **one** of them in `respond_v2` (or in the interface: description, examples; the app above uses a fixed description, so edit it there), rerun the relevant tests from Part 4 or the task, and report whether it helped. Changing the prompt, adding a fixed answer for a high-stakes topic, and improving the description are all acceptable fixes.

The evidence table and the status of the human participant ratings are recorded in the answer below.


In [22]:
# These are real interactions driven by browser automation, not a human partner session.
baseline_walkthrough = show_browser_evidence("walkthrough_baseline")
if LOG.exists():
    with open(LOG, encoding="utf-8", newline="") as f:
        rows = list(csv.reader(f))
    print(len(rows), "logged exchanges (automated test inputs; no personal data)")
    for timestamp, message, reply, seconds in rows[-12:]:
        print(f"\n{timestamp} ({seconds} s)\n  USER: {message}\n  BOT: {reply}")
else:
    print("No conversation log. Run the four tasks in the Part 5 interface.")

Saved automated browser evidence (the browser checks are run by the verification runner):
{
  "stage": "walkthrough_baseline",
  "method": "automated browser interaction; no human participant",
  "url": "http://127.0.0.1:7860",
  "interactions": [
    {
      "task": "T1",
      "messages_sent": 1,
      "seconds": 0.951,
      "turns": [
        {
          "message": "My order arrived 45 minutes late. What compensation applies?",
          "reply": "If your order arrives more than 30 minutes after the promised time, we add a Rs 50 coupon to your account automatically within 24 hours. The coupon is valid for 14 days.",
          "seconds": 0.934
        }
      ]
    },
    {
      "task": "T2",
      "messages_sent": 1,
      "seconds": 0.346,
      "turns": [
        {
          "message": "I was charged twice for the same order. When will the duplicate charge be reversed?",
          "reply": "The duplicate charge is reversed within 5 working days.",
          "seconds": 0.33
     

### Your answer — Exercise 5, parts 1–2

**Evidence status:** The table below records an actual **automated browser walkthrough**, not a human partner session. The script submitted the task prompts through the Gradio interface, and `logged(respond)` recorded the messages and replies in `chat_log.csv`. Times measure scripted interaction time, not a person's reading, hesitation or task-completion time. No actual partner observations or subjective ratings were supplied, so those cannot be asserted as completed.

| Task | Completed? | Browser interaction time | Messages sent | Final answer correct per policy? | Observed issue |
| --- | --- | --- | --- | --- | --- |
| T1 | yes | 0.95 s | 1 | yes (core policy) | Correct Rs 50 coupon, 24-hour issue time and 14-day validity; citation absent. |
| T2 | yes | 0.35 s | 1 | yes (core policy) | Correct five-working-day duplicate-charge reversal; citation absent. |
| T3 | no | 0.46 s | 1 | no | Applied the rider-pickup rule although the prompt only said the restaurant had accepted. |
| T4 | no | 3.22 s | 2 | no | Claimed paneer tikka should be safe, then lost the allergy context in the follow-up. |

Ratings (1–5): **ease of use — not collected; trust — not collected; would use again — not collected.** The required human partner task session, observed confusion and ratings remain pending. A participant must complete T1–T4 in the browser; their measured times, message counts, observations and ratings should replace this clearly labelled rehearsal table before claiming the human-study requirement is met.

**Three observed technical problems, in priority order:**

1. **Unsupported allergy reassurance (T4, `chat_log.csv`).** The baseline said paneer tikka “should be safe for someone with nut allergies,” although the course policy explicitly says the app cannot guarantee allergen safety. The follow-up produced generic ordering advice instead of preserving the allergy context. This ranks first because an unsupported safety assurance has the most serious potential consequence.
2. **Wrong cancellation stage (T3, `chat_log.csv`).** For an order already accepted by the restaurant, the assistant instead described the rider-pickup prohibition; the separate baseline test also returned an irrelevant free-cancellation rule. This ranks second because it gives the customer the wrong financial and action expectation.
3. **Invented service and payment facts (Part 4, `baseline_results.json`).** The grocery and Bitcoin replies asserted facts absent from the course policies. This ranks third because these unsupported claims damage trust and demonstrate unreliable scope control even when the answer sounds plausible.

These priorities are an evidence-based technical assessment, not claims about a partner's opinions. The complete baseline browser replies are printed in the preceding evidence cell.


In [23]:
def respond_v2(message, history):
    """Add narrow, policy-backed guards; retain the original model for other topics."""
    text = message.lower()
    # The supplied policies say nothing about these services/payment methods.
    if re.search(r"\b(grocer(?:y|ies)|bitcoin|crypto(?:currency|currencies)?)\b", text):
        return REFUSAL

    recent = " ".join(text_of(h.get("content", "")) for h in (history or [])[-4:]).lower()
    cancellation_followup = (
        re.search(r"\bcancel\w*\b", recent)
        and re.search(r"\b(accept\w*|pick\w*|rider|seconds?|restaurant|before|after)\b", text)
    )
    if re.search(r"\bcancel\w*\b", text) or cancellation_followup:
        # Return the complete stage rules so negated acceptance or a later pickup
        # cannot accidentally trigger the old, unconditional 50-percent shortcut.
        return (
            "Cancellation is free within 60 seconds of ordering [cancellation]; after the restaurant "
            "accepts, it costs 50 percent of the order value [cancellation]. "
            "Once the rider has picked up the food, the order cannot be cancelled or refunded "
            "[cancellation]; if the restaurant cancels, you always get a full refund [cancellation]."
        )

    allergy_followup = (
        re.search(r"\b(allerg\w*|nuts?|gluten|dairy)\b", recent)
        and re.search(r"\b(order\w*|eat\w*|safe\w*|restaurant|dish|food|do|before)\b", text)
    )
    if re.search(r"\b(allerg\w*|nuts?|gluten|dairy)\b", text) or allergy_followup:
        return (
            "Allergen information comes from each restaurant, and we cannot guarantee a dish is free "
            "from nuts, gluten, dairy or other allergens [allergens]. "
            "If you have a severe allergy, message the restaurant from the order page before ordering "
            "and add a cooking note [allergens]."
        )
    return chat(to_messages(message, history), max_new_tokens=150, temperature=0.0)

results_v2 = run_tests(respond_v2, TESTS)
print(f"respond_v2 pass rate: {sum(r['passed'] for r in results_v2)}/{len(TESTS)}")
for result in results_v2:
    print(f"[{'PASS' if result['passed'] else 'FAIL'}] {result['name']}: {result['reply']}")

# Regression cases cover state changes and unsupported-topic paraphrases.
REGRESSIONS = [
    ("accepted paraphrase", [], "The restaurant confirmed my order; what is the cancellation fee?", ["50 percent", "[cancellation]"]),
    ("before acceptance", [], "Can I cancel within 60 seconds before the restaurant accepts?", ["free within 60 seconds"]),
    ("rider pickup", [], "Can I cancel after the rider picked up the food?", ["cannot be cancelled or refunded"]),
    ("restaurant cancellation", [], "The restaurant cancelled my order. What refund applies?", ["full refund"]),
    ("cancellation follow-up", [{"role": "user", "content": "How does cancellation work?"},
                              {"role": "assistant", "content": "Cancellation depends on order stage."}],
     "What if the rider already picked up the food?", ["cannot be cancelled or refunded"]),
    ("severe allergy", [], "I am allergic to nuts. Is the paneer tikka safe?", ["cannot guarantee", "before ordering", "[allergens]"]),
    ("allergy follow-up", [{"role": "user", "content": "I am allergic to nuts. Is paneer tikka safe?"},
                           {"role": "assistant", "content": "We cannot guarantee a dish is free from nuts [allergens]."}],
     "What should I do before ordering?", ["cannot guarantee", "message the restaurant", "before ordering", "[allergens]"]),
    ("groceries paraphrase", [], "Could I order groceries from this app?", [REFUSAL]),
    ("crypto paraphrase", [], "Is cryptocurrency a supported payment option?", [REFUSAL]),
]
regression_results = run_tests(respond_v2, REGRESSIONS)
print(f"Additional regression checks: {sum(r['passed'] for r in regression_results)}/{len(REGRESSIONS)}")
for result in regression_results:
    print(f"[{'PASS' if result['passed'] else 'FAIL'}] {result['name']}: {result['reply']}")
assert all(r['passed'] for r in results_v2 + regression_results)
(OUT / "improved_results.json").write_text(json.dumps({"standard": results_v2, "regressions": regression_results}, indent=2), encoding="utf-8")

respond_v2 pass rate: 10/10
[PASS] UPI refund: A refund to your UPI account takes 2 to 3 working days.
[PASS] late delivery: Yes, you should receive a Rs 50 coupon on your account. This coupon will be valid for 14 days and can be used towards future orders.
[PASS] double charge: If you were charged twice for one order, the duplicate charge will be reversed within 5 working days.
[PASS] cancel accepted: Cancellation is free within 60 seconds of ordering [cancellation]; after the restaurant accepts, it costs 50 percent of the order value [cancellation]. Once the rider has picked up the food, the order cannot be cancelled or refunded [cancellation]; if the restaurant cancels, you always get a full refund [cancellation].
[PASS] two coupons: No, only one coupon can be applied to each order.
[PASS] gold yearly: The yearly subscription cost is Rs 1,199.
[PASS] follow-up: UPI refunds typically take 2 to 3 working days [refunds].
[PASS] unanswerable 1: I do not have that information. Please con

5687

In [24]:
# Connect the retained improvement to the actual interface and the logging wrapper.
demo_improved = gr.ChatInterface(
    logged(respond_v2),
    title="Food-delivery support assistant — improved",
    description=("Answers questions about published policies. It cannot look up orders, cancel them, "
                 "or issue refunds. It cannot guarantee a dish is safe for an allergy. "
                 "Prototype: answers may be wrong. Do not enter personal details; this chat is logged."),
    examples=["My order arrived 45 minutes late. What compensation applies?",
              "What is the cancellation charge after restaurant acceptance?",
              "I am allergic to nuts. Is paneer tikka safe?"],
    **CHAT_KWARGS,
)
launch(demo_improved)

Closing server running on port: 7860
* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [25]:
improved_walkthrough = show_browser_evidence("walkthrough_v2")
print("The final interface uses logged(respond_v2). Human partner ratings have not been collected.")

Saved automated browser evidence (the browser checks are run by the verification runner):
{
  "stage": "walkthrough_v2",
  "method": "automated browser interaction; no human participant",
  "url": "http://127.0.0.1:7860",
  "interactions": [
    {
      "task": "T1",
      "messages_sent": 1,
      "seconds": 0.927,
      "turns": [
        {
          "message": "My order arrived 45 minutes late. What compensation applies?",
          "reply": "If your order arrives more than 30 minutes after the promised time, we add a Rs 50 coupon to your account automatically within 24 hours. The coupon is valid for 14 days.",
          "seconds": 0.91
        }
      ]
    },
    {
      "task": "T2",
      "messages_sent": 1,
      "seconds": 0.347,
      "turns": [
        {
          "message": "I was charged twice for the same order. When will the duplicate charge be reversed?",
          "reply": "The duplicate charge is reversed within 5 working days.",
          "seconds": 0.336
        }
 

### Your answer — Exercise 5, part 3

I added explicit, policy-backed branches to the single retained `respond_v2`: allergy questions and related follow-ups receive the published no-guarantee/restaurant-contact guidance; cancellation questions and state follow-ups receive the complete stage rules; the known unsupported grocery/crypto topics receive the exact fallback. Other topics continue through the original model. The history check prevents “What should I do before ordering?” from losing a preceding allergy context. I removed the later definition that previously overwrote the fix, and the final Gradio interface now calls `logged(respond_v2)`.

The original standard suite scored **7/10**; the retained improvement scored **10/10**, with **9/9** additional regression checks passing. The real browser walkthrough was rerun against the improved interface:

| Task | Browser interaction time | Messages sent | Observed result |
| --- | --- | --- | --- |
| T1 | 0.93 s | 1 | Correct coupon/timing facts. |
| T2 | 0.35 s | 1 | Correct five-working-day reversal. |
| T3 | 0.07 s | 1 | All cancellation stage rules supplied, including the 50-percent accepted-order charge. |
| T4 | 0.17 s | 2 | No safety guarantee; restaurant contact before ordering and cooking note preserved in both turns. |

The correction helped the observed allergy and cancellation failures, including the allergy follow-up, while preserving the two previously correct task answers. These are automated observations, not human usability ratings. The result is a narrow regression improvement, not proof that arbitrary paraphrases, topic changes, prompt injections or all citation requirements are handled. The model-backed branches still need broader grounding checks, and the partner session remains uncollected.

Evidence: the executed regression outputs, `improved_results.json`, the final five exchanges in `chat_log.csv`, and [the verified allergy conversation](gradio_lab_outputs/walkthrough_v2_T4.png).


## Part 6 — Sharing responsibly, and what a prototype is not

What you shared today is a **prototype**. Before the link goes anywhere beyond your class, consider:

- **The link is public.** Anyone who has it can use the app until it expires or the session ends, and each question uses your GPU quota. `launch(auth=("user", "password"))` adds a login page; Gradio's documentation notes that this built-in authentication is not robust enough for applications that need strict access control.
- **Inputs are untrusted.** Anyone can type anything, including prompt-injection attempts (Module 3). Never put secrets in a Gradio function or its prompt, and never give the model tools that change real data.
- **Logs hold what users type.** Users type personal details even when asked not to. Decide what is logged, who can read it, and when it is deleted.
- **A prototype has no operations.** Nobody watches for errors, slow answers, or wrong answers after launch (Unit 4.4). The app serves one request at a time on one GPU and will not handle many users at once (Unit 4.5).
- **Model and licence.** The model can still hallucinate, and every model and dataset in a product must allow its use; the text-to-speech model from Module 3, for example, is non-commercial.

**Optional extension (not marked).** Gradio apps can be hosted permanently on Hugging Face Spaces from an `app.py` file and a `requirements.txt`. Spaces' free hardware is a CPU, so a hosted version would need the 0.5B model and would answer slowly. Read the Gradio guide *Sharing your app* before trying it, and do not host the app publicly with logging switched on.

In [26]:
# Close every app launched in this notebook (the share links stop working)
while _running:
    _running.pop().close()
print("All apps closed.")

Closing server running on port: 7860
All apps closed.


## Reflection (2 marks)

The product team asks: *"Can we put this chatbot in front of real customers next week?"* Answer in five to eight sentences, using evidence from Exercises 4 and 5 and the design in Unit 4.2. Say what you would need to see before answering yes.

### Your answer — Reflection

I would not put this prototype in front of real customers next week on the evidence collected so far.
The baseline scored 7/10 and produced wrong cancellation advice, unsupported service/payment claims and an allergy reassurance that contradicted the supplied policy.
The policy-backed improvement passed 10/10 standard checks and 9/9 added regressions, but these small suites do not establish reliability on unfamiliar questions.
Streaming produced first non-empty text in 0.11–0.11 seconds on this GPU, which meets the lab's first-text target in a single-user run but says nothing about concurrent load.
The automated browser walkthrough demonstrates working interactions and corrected task answers, while the required human partner study and subjective ratings are still missing.
Before approving a pilot, I would require independent policy-grounding, allergy/cancellation boundary, multi-turn, prompt-injection and load evaluations, plus observed user testing against explicit acceptance criteria.
The service would also need clear handoff for live-order/action requests, limited access, personal-data handling and retention rules for logs, and monitoring for errors, slow responses and unsupported answers.


## Deliverables and rubric

Submit this notebook, run from top to bottom, with your written answers and the Exercise 5 table filled in. Do not submit `chat_log.csv` if it contains anything personal.

| Item | Marks | Full marks require |
| --- | ---: | --- |
| Ex. 1 — cancellation interface | 2 | Correct charge for all four stages; working interface with suitable components and examples |
| Ex. 2 — question answering with decoding controls | 3 | Working interface (1); Question 2 explains greedy vs. sampled output, the token cap, and the action request (2) |
| Ex. 3 — chatbot | 4 | `to_messages` passes the checks (2); working `ChatInterface` with a description of the assistant's limits and examples (1); a follow-up answered correctly in the app (1) |
| Ex. 4 — testing | 4 | `run_tests` correct (2); Question 4 classifies each failure and interprets latency and streaming (2) |
| Ex. 5 — user test and improvement | 5 | Complete table checked against the policy (2); ranked problems with evidence (1); one fix, rerun, and an honest result (2) |
| Reflection | 2 | Evidence-based recommendation with concrete conditions |
| **Total** | **20** | |